## Tabella ordini

In [1]:
import pandas as pd

ordini = pd.read_csv("../data/olist_orders_dataset.csv")
ordini.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [2]:
ordini.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB


In [3]:
ordini["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

### Cosa ho scoperto: struttura e valori mancanti

- 99.441 ordini, 8 colonne
- Le date sono lette come testo: vanno convertite in date
- Mancano alcune date (approvazione, corriere, consegna):
  corrispondono agli ordini non consegnati, quindi non sono errori
- Piccola incoerenza di 2 ordini tra stato e data di consegna: da verificare in SQL

In [4]:
ordini["order_purchase_timestamp"] = pd.to_datetime(ordini["order_purchase_timestamp"])
ordini["order_approved_at"] = pd.to_datetime(ordini["order_approved_at"])
ordini["order_delivered_carrier_date"] = pd.to_datetime(ordini["order_delivered_carrier_date"])
ordini["order_delivered_customer_date"] = pd.to_datetime(ordini["order_delivered_customer_date"])
ordini["order_estimated_delivery_date"] = pd.to_datetime(ordini["order_estimated_delivery_date"])

In [5]:
ordini.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  str           
 1   customer_id                    99441 non-null  str           
 2   order_status                   99441 non-null  str           
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   order_approved_at              99281 non-null  datetime64[us]
 5   order_delivered_carrier_date   97658 non-null  datetime64[us]
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  99441 non-null  datetime64[us]
dtypes: datetime64[us](5), str(3)
memory usage: 6.1 MB


In [6]:
print(ordini["order_purchase_timestamp"].min())
print(ordini["order_purchase_timestamp"].max())

2016-09-04 21:15:19
2018-10-17 17:30:18


In [7]:
ordini["order_purchase_timestamp"].dt.to_period("M").value_counts().sort_index()

order_purchase_timestamp
2016-09       4
2016-10     324
2016-12       1
2017-01     800
2017-02    1780
2017-03    2682
2017-04    2404
2017-05    3700
2017-06    3245
2017-07    4026
2017-08    4331
2017-09    4285
2017-10    4631
2017-11    7544
2017-12    5673
2018-01    7269
2018-02    6728
2018-03    7211
2018-04    6939
2018-05    6873
2018-06    6167
2018-07    6292
2018-08    6512
2018-09      16
2018-10       4
Freq: M, Name: count, dtype: int64

### Cosa ho scoperto: periodo coperto dai dati

- Gli ordini vanno da settembre 2016 a ottobre 2018
- I primi 4 mesi sono incompleti (novembre 2016 non ha nessun ordine), così come gli ultimi due
- Esempio: agosto 2018 ha circa 6512 ordini, settembre 2018 solo 16 → è un taglio dei dati, non un calo delle vendite
- Probabile causa: il dataset è stato estratto a metà ottobre 2018
- Per l'analisi userò il periodo da gennaio 2017 ad agosto 2018 (20 mesi completi)

## Tabella clienti

In [8]:
clienti = pd.read_csv("../data/olist_customers_dataset.csv", dtype={"customer_zip_code_prefix": str})
clienti.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,08775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [9]:
clienti.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  str  
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: str(5)
memory usage: 3.8 MB


In [10]:
len(clienti)

99441

In [11]:
clienti["customer_id"].nunique()

99441

In [12]:
clienti["customer_unique_id"].nunique()

96096

### Cosa ho scoperto

- `customer_id` ha 99.441 valori unici, uno per ogni ordine: Olist crea un nuovo `customer_id` a ogni acquisto, quindi non identifica la persona.
- `customer_unique_id` identifica la persona e ha 96.096 valori unici. La differenza (3.345) sono ordini fatti da clienti che avevano già comprato: solo il 3,4% circa degli ordini è un riacquisto.
- Nessun valore mancante: tutte e 5 le colonne sono complete.
- Il CAP (`customer_zip_code_prefix`) è caricato come testo: è un codice, non un numero su cui fare calcoli, e come numero perderebbe gli zeri iniziali (es. `01151` diventerebbe `1151`).

## Tabella righe d'ordine

In [13]:
righe_ordine = pd.read_csv("../data/olist_order_items_dataset.csv")
righe_ordine.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [14]:
righe_ordine.info()

<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  str    
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  str    
 3   seller_id            112650 non-null  str    
 4   shipping_limit_date  112650 non-null  str    
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), str(4)
memory usage: 6.0 MB


In [15]:
len(righe_ordine)

112650

In [16]:
righe_ordine["order_id"].nunique()

98666

In [17]:
righe_ordine.describe()

,order_item_id,price,freight_value
count,112650.000000,112650.000000,112650.000000
mean,1.197834,120.653739,19.990320
std,0.705124,183.633928,15.806405
min,1.000000,0.850000,0.000000
25%,1.000000,39.900000,13.080000
50%,1.000000,74.990000,16.260000
75%,1.000000,134.900000,21.150000
max,21.000000,6735.000000,409.680000


### Cosa ho scoperto

- Ogni riga è un prodotto dentro un ordine: 112.650 righe per 98.666 ordini, perché un ordine può contenere più prodotti.
- La tabella ordini ha 99.441 ordini, quindi 775 ordini non hanno nessun prodotto. Ipotesi: sono soprattutto ordini in stato `unavailable` (prodotto non disponibile). Da verificare in SQL.
- Non esiste una colonna quantità: ogni pezzo è una riga a sé. Per contare i pezzi venduti, per prodotto o per categoria, si contano le righe.
- Nessun valore mancante in nessuna colonna.
- Prezzo: la media (circa 121) è molto più alta della mediana (circa 75), perché pochi prodotti molto cari (fino a 6.735) tirano su la media. Per descrivere il prezzo tipico uso la mediana.

## Tabella prodotti

In [18]:
prodotti = pd.read_csv("../data/olist_products_dataset.csv")
prodotti.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [19]:
prodotti.info()

<class 'pandas.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  str    
 1   product_category_name       32341 non-null  str    
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), str(2)
memory usage: 2.3 MB


In [20]:
len(prodotti)

32951

In [21]:
prodotti["product_id"].nunique()

32951

In [22]:
prodotti["product_category_name"].nunique()

73

In [23]:
prodotti.isna().sum()

product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

In [24]:
righe_ordine["product_id"].nunique()

32951

### Cosa ho scoperto

- Ogni riga è un prodotto del catalogo: 32.951 righe e 32.951 `product_id` unici, quindi nessun duplicato.
- Ci sono 73 categorie, scritte in portoghese: servirà la tabella di traduzione.
- 610 prodotti (meno del 2%) non hanno la categoria. Agli stessi 610 mancano anche lunghezza del nome, lunghezza della descrizione e numero di foto: probabilmente sono schede prodotto lasciate incomplete.
- Nella tabella righe d'ordine ci sono 32.951 prodotti diversi, lo stesso numero del catalogo: forte indizio che ogni prodotto sia stato venduto almeno una volta. Quindi i 610 non sono prodotti inattivi. Da confermare in SQL.
- Scelta: i prodotti senza categoria restano nell'analisi con l'etichetta "senza categoria", così il fatturato totale non perde pezzi. L'etichetta la applico in SQL.
- A 2 prodotti mancano peso e dimensioni (lunghezza, altezza, larghezza): alle nostre domande non servono.

## Tabella traduzione categorie

In [25]:
traduzioni = pd.read_csv("../data/product_category_name_translation.csv")
traduzioni.head()

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


In [26]:
traduzioni.info()

<class 'pandas.DataFrame'>
RangeIndex: 71 entries, 0 to 70
Data columns (total 2 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   product_category_name          71 non-null     str  
 1   product_category_name_english  71 non-null     str  
dtypes: str(2)
memory usage: 1.2 KB


In [27]:
len(traduzioni)

71

### Cosa ho scoperto

- 71 categorie tradotte su 73: a 2 categorie manca la traduzione in inglese.
- Collegando prodotti e traduzioni, i prodotti di quelle 2 categorie sparirebbero (INNER JOIN) o resterebbero senza nome inglese (LEFT JOIN).
- Scelta: in SQL individuo le 2 categorie e le traduco a mano. Tradurre un nome noto non è inventare un dato.
- Nessun valore mancante.

## Tabella recensioni

In [28]:
recensioni = pd.read_csv("../data/olist_order_reviews_dataset.csv")
recensioni.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


In [29]:
recensioni.info()

<class 'pandas.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   review_id                99224 non-null  str  
 1   order_id                 99224 non-null  str  
 2   review_score             99224 non-null  int64
 3   review_comment_title     11568 non-null  str  
 4   review_comment_message   40977 non-null  str  
 5   review_creation_date     99224 non-null  str  
 6   review_answer_timestamp  99224 non-null  str  
dtypes: int64(1), str(6)
memory usage: 5.3 MB


In [30]:
print(len(recensioni))
print(recensioni["review_id"].nunique())
print(recensioni["order_id"].nunique())

99224
98410
98673


In [31]:
recensioni["review_score"].value_counts().sort_index()

review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64

### Cosa ho scoperto

- 99.224 recensioni, ma solo 98.673 ordini diversi: alcuni ordini hanno più di una recensione.
- Anche i `review_id` unici sono meno delle righe (98.410): la stessa recensione può comparire su più ordini.
- Rischio: collegando recensioni e ordini, un ordine con 2 recensioni verrebbe contato due volte, e il suo fatturato raddoppierebbe.
- Scelta: in SQL tengo una sola recensione per ordine, la più recente.
- I voti sono sbilanciati: circa il 58% è 5 stelle, e il voto 1 (circa l'11%) è più frequente del 2 e del 3.
- I commenti mancano nella maggior parte delle recensioni: non è un errore, molti clienti danno solo il voto. Per la domanda 4 serve solo il voto, quindi i commenti non li uso.